In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import scikit_posthocs as sp

print("Все библиотеки успешно импортированы!")

# Task 1 - Первичный осмотр и очистка данных

In [ ]:
df = pd.read_csv('data/StudentsPerformance.csv')

In [ ]:
print(f'Число строк датасета = {df.shape[0]}, колличество столбцов = {df.shape[1]}')
print('----------------------------------------------------')
print('Столбцы и тыпы данных их значений:')
print(df.dtypes)
print('----------------------------------------------------')
print(f'Количество пропущенных значений = {df.isna().sum().sum()}')
print('----------------------------------------------------')
print('Вид датасета (первые 5 элементов):')
print(df.head())
print('----------------------------------------------------')
print('Вид датасета (последние 5 элементов):')
print(df.tail())

# Создаём тестовый, неочищенный датасет
df_not_clean = df.copy()

In [ ]:
# Проверим наличие дубликатов и в случае нахождения - удалим их

before_clean_d = df_not_clean.shape[0]  # Помещаем в переменную колличество строк из тестового (неочищенного) блока данных, созданного в верхнем блоке.
df = df.drop_duplicates().reset_index(drop=True)  # Чистим наш основной data-set, если он содержит дубликаты - они будут удалены и индексы будут рассчитаны заново
after_clean_d = df.shape[0] # Помещаем в переменную колличество строк из основного блока данных, после удаления дубликатов (если они были найдены).
if before_clean_d == after_clean_d:
  print('Data-set не содержит дубликатов')
else:
  print(f'Размер датасета, до удаления строк-дубликатов: {before_clean_d}')
  print('----------------------------------------------------')
  print(f'Размер датасета, после удаления строк-дубликатов: {after_clean_d}')
  print('----------------------------------------------------')
  print(f'Удалено строк: {before_clean_d - after_clean_d}')
  print(df.head())

In [ ]:
# Проверим оценки на аномалии (Не должны привышать 100 и быть ниже 0)
def check_anomalies(data_series, min_val = 0, max_val = 100):
  anomalies = len(data_series[(data_series > max_val) | (data_series < min_val)])
  print(f'Аномальных значений в столбце {data_series.name} найдено: {anomalies}')

check_anomalies(df['math score'])
check_anomalies(df['reading score'])
check_anomalies(df['writing score'])

In [ ]:
# Вывод: Баллы студентов корректны, никаких действий применять не следует.

In [ ]:
# Теперь проведём проверку столбцов с категориальными значениями,
# чтобы в них не было ошибочных данных
def check_categories(data_series):
  print(data_series.unique())

check_categories(df['parental level of education'])
check_categories(df['gender'])  # Может быть 'female' или 'male'
check_categories(df['lunch']) # Может быть 'standard' или 'free/reduced'
check_categories(df['test preparation course']) # Может быть 'none' или 'completed'

In [ ]:
# Вывод: Столбцы с категориальными значениями - корректны.

In [ ]:
# Так как в наших данных не встречаются пропущенные значения, то
# обработка пропусков не требуется, но давайте представим что они всё же
# присутствуют; для дальнейшей аналитики нам потребуется значение блока - 'test preparation course'
# и блоков с оценками - такими как: 'math score', 'reading score', 'writing score'.
# В случае, если пропущено значение в столбце 'test preparation course' -
# мы будем удалять всю строку, а в случае если неизвестна какая-либо
# оценка из теста - мы будем подставлять средний балл, в зависимости от того, к какой
# группе относится студент (Студент прошедший тест - получит среднее от оценки других таких же студентов, прошедших тест,
# а студент, не прошедший тест, соответственно получит балл из среднего другой группы)

In [ ]:
df = df.dropna(subset=['test preparation course']).reset_index(drop=True)  # Удаляем всю строку, если в поле 'test preparation course' отсутствуют данные, а затем обнуляем индекс.

# Создадим список с названиями столбцов с результатами учеников
discipline_result = ['math score', 'reading score', 'writing score']

# Заполнение пропусков средним по группе:
def fill_gapes(group, discipline_mean):
  return df.groupby(group)[discipline_mean].transform(lambda x: x.fillna(x.mean()))

df['math score'] = fill_gapes('test preparation course', 'math score')
df['reading score'] = fill_gapes('test preparation course', 'reading score')
df['writing score'] = fill_gapes('test preparation course', 'writing score')

In [ ]:
# Финальная проверка
print(f"Итоговый размер: {df.shape}")
print(f"Пропусков: {df.isna().sum().sum()}")
print(f"Дубликатов: {df.duplicated().sum()}")
df.info()

## Task 2 - Исследовательский анализ данных (EDA). (Поиск закономерностей и зависимостей)

In [ ]:
# 0 Определим максимальный и минимальный балл по каждой дисциплине
result = df[discipline_result].agg(['min', 'max'])
result.head()

In [ ]:
# Мы наблюдаем, что самый минимальный балл был получен по дисциплине math, таким образом,
# сделаем предположение, что математика - является наиболее сложной дисциплиной для сдачи.

In [ ]:
# 1 Определим средний балл парней и девушек по всем дисциплинам:
def mean_by_group(condition, discipline_result):
  return df.groupby(condition)[discipline_result].mean()

math_score_result_by_gender = mean_by_group('gender', 'math score')
reading_score_result_by_gender = mean_by_group('gender', 'reading score')
writing_score_result_by_gender = mean_by_group('gender', 'writing score')

# Создаём график - размером 2X2, из которых нижний правый столбец не будет задействован
fig, axs = plt.subplots(2,2,figsize=(12, 8))
fig.suptitle('Средние баллы студентов по полу', fontsize=16, fontweight='bold')

bars1 =axs[0,0].bar(math_score_result_by_gender.index, math_score_result_by_gender.values, color=['skyblue', 'lightcoral']) # Выбираем позицию графика и берём информацию для построения из 1-й переменной, созданной выше.
axs[0, 0].bar_label(bars1, fmt='%.1f')  # Добавляем подпись
axs[0, 0].set_title('Предмет: math')  # Добавляем надпись
axs[0, 0].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray') # Добавляем сетку с таблице и оформляем её

bars2 = axs[0,1].bar(reading_score_result_by_gender.index, reading_score_result_by_gender.values, color=['skyblue', 'lightcoral'])
axs[0, 1].bar_label(bars2, fmt='%.1f')
axs[0, 1].set_title('Предмет: reading')
axs[0, 1].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

bars3 = axs[1,0].bar(writing_score_result_by_gender.index, writing_score_result_by_gender.values, color=['skyblue', 'lightcoral'])
axs[1, 0].bar_label(bars3, fmt='%.1f')
axs[1, 0].set_title('Предмет: writing')
axs[1, 0].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

# Скрываем последний пустой график
axs[1, 1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# 2 Проверим - влияет ли прохождение подготовительных курсов на экзаменационный балл:

# Как и в прошлом примере, узнаем средний балл по каждому предмету, но уже в зависимости от прохождения курсов
math_score_result_course_influence = mean_by_group('test preparation course', 'math score')
reading_score_result_course_influence = mean_by_group('test preparation course', 'reading score')
writing_score_result_course_influence = mean_by_group('test preparation course', 'writing score')


# Теперь, узнаем общий средний балл по совокупности предметов:
# Соединим все наши Series (Series_1, Series_2. Series_3) - в одну общую таблицу (DataFrame)
df_tests_score_result_course_influence = pd.concat([math_score_result_course_influence, reading_score_result_course_influence, writing_score_result_course_influence], axis=1)
# Обнулим индекс, чтобы при нахождении среднего арифметического - значения были корректны
df_tests_score_result_course_influence = df_tests_score_result_course_influence.reset_index()
# Найдём среднее арифметическое по совокупности предметов и добавим результат в отдельный столбец со значением - mean
df_tests_score_result_course_influence['mean'] = df_tests_score_result_course_influence[['math score','reading score','writing score']].mean(axis=1)
# Посмотрим наш DataFrame
df_tests_score_result_course_influence.head()


In [ ]:
# Переходим к визуализации результатов:
fig, axs = plt.subplots(2,2,figsize=(12, 8))
fig.suptitle('Средние баллы студентов, прошедших/не прошедших подготовительные курсы', fontsize=16, fontweight='bold')

bars1 =axs[0,0].bar(math_score_result_course_influence.index,
                    math_score_result_course_influence.values,
                    color=['skyblue', 'lightcoral'])
axs[0, 0].bar_label(bars1, fmt='%.1f')
axs[0, 0].set_title('Предмет: math')
axs[0, 0].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

bars2 = axs[0,1].bar(reading_score_result_course_influence.index, reading_score_result_course_influence.values, color=['skyblue', 'lightcoral'])
axs[0, 1].bar_label(bars2, fmt='%.1f')
axs[0, 1].set_title('Предмет: reading')
axs[0, 1].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

bars3 = axs[1,0].bar(writing_score_result_course_influence.index, writing_score_result_course_influence.values, color=['skyblue', 'lightcoral'])
axs[1, 0].bar_label(bars3, fmt='%.1f')
axs[1, 0].set_title('Предмет: writing')
axs[1, 0].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

# Т.к. для создания данного графика мы используем DataFrame, а не Series, то для визуализации использум не значения .index и .value, а непосредственно названия столбцов,
# которые используем для визуализации.
bars4 = axs[1,1].bar(df_tests_score_result_course_influence['test preparation course'], df_tests_score_result_course_influence['mean'], color=['skyblue', 'lightcoral'])
axs[1, 1].bar_label(bars4, fmt='%.1f')
axs[1, 1].set_title('Общий средний балл')
axs[1, 1].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')


plt.tight_layout()
plt.show()


In [ ]:
# Вывод: Студенты, прошедшие подготовительный курс, имеют более высокий балл, чем те, которые его не прошли. Таким образом,
# делаем вывод, что подготовительные курсы - оказывают ощутимое влияние на итоговый результат студентов.

In [ ]:
# 3 Просмотрим распределение оценок по среднему баллу по предмету: math:
# Данный предмет был выбран т.к. по сравнению с другими дисциплинами, результаты по математики самые низкие, на что указывает
# верхняя диаграмма результатов, а также самый первый тест(0), где минимальный балл по математики ниже, чем у других дисциплин.
# Просмотрим на распределение данных по данному предмету и сравним полученные значения с результатами
# по другим дисциплинам
fig, ax = plt.subplots(figsize=(12, 8))
fig.suptitle('Распределение оценок между студентами по предмету: math', fontsize=16, fontweight='bold')
bins = np.linspace(0, 100, 21)  # 20 бинов от 0 до 100 с шагом 5
ax.hist(df['math score'], bins=bins, edgecolor='black', linewidth=1.5, color='skyblue')
ax.set_xlabel('Баллы за экзамене', fontsize=12)
ax.set_ylabel('Количество студентов', fontsize=12)
ax.grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Вывод: Распределение оценок по математике имеет колоколообразную форму
# с центром в диапазоне 50-70 баллов. Это подтверждает, что математика
# является наиболее сложным предметом — основная масса студентов
# демонстрирует средний уровень знаний, при этом присутствует небольшая
# группа отстающих (0-40 баллов) и ограниченное количество отличников (90-100 баллов).

In [ ]:
# Проверим распределение по другим дисциплинам:
fig, axs = plt.subplots(2,1, figsize=(12, 10))
fig.suptitle('Распределение оценок между студентами по предметам: reading, writing', fontsize=16, fontweight='bold')

axs[0].hist(df['reading score'], bins=bins, edgecolor='black', linewidth=1.5, color='skyblue')
axs[0].set_xlabel('Баллы за экзамене "reading"', fontsize=12)
axs[0].set_ylabel('Количество студентов', fontsize=12)
axs[0].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

axs[1].hist(df['writing score'], bins=bins, edgecolor='black', linewidth=1.5, color='skyblue')
axs[1].set_xlabel('Баллы за экзамене "writing"', fontsize=12)
axs[1].set_ylabel('Количество студентов', fontsize=12)
axs[1].grid(True, alpha=0.4, axis='y', linestyle='--', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Таким образом, первоначальный выбор математики как «самого слабого» предмета визуально подтверждается:
# центр её распределения действительно левее, а форма более «приземистая» по сравнению с более высокими
# и смещёнными вправо распределениями reading и writing.

In [ ]:
# 4 В качестве заключительного этапа - проверим влияние образования родителей на итоговую оценку студента:
# Для начала, просмотрим ещё раз (данное действие мы уже выполняли, при первичном осмотре данных), какие значения может содержать столбец 'parental level of education'.
print(df['parental level of education'].unique())

In [ ]:
# Мы видим 5 позиций, применим фильтр и выедем средние оценки по каждому предмету, в зависимости от уровня образования родителей.
result_depen_of_parents_education = df.groupby('parental level of education').agg({'math score':'mean','reading score':'mean','writing score':'mean'})

In [ ]:
result_depen_of_parents_education.head()

In [ ]:
# Визуализируем результаты для наглядности:

In [ ]:
# Упорядочим наши данные с повышением уровней образования родителей
education_order = [
    'high school',
    'some high school',
    'some college',
    "associate's degree",
    "bachelor's degree",
    "master's degree"
]
result_ordered = result_depen_of_parents_education.reindex(education_order)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(result_ordered.index, result_ordered['math score'], color='#2196F3', label='math score', marker='o', markersize=8, linewidth=2)
ax.plot(result_ordered.index, result_ordered['reading score'], color='#4CAF50', label='reading score', marker='s', markersize=8, linewidth=2)
ax.plot(result_ordered.index, result_ordered['writing score'], color='#FF5722', label='writing score', marker='^', markersize=8, linewidth=2)
ax.legend()
plt.show()

In [ ]:
# Вывод: cуществует социально-образовательное неравенство: студенты, чьи родители имеют высшее образование (особенно магистерскую степень),
# показывают значительно лучшие академические результаты. Особенно это проявляется в гуманитарных науках, таких как 'reading' и 'writing'.

# Task 3 - Статистический анализ и проверка гипотез

In [ ]:
 # Проверим ряд статистических гипотез, которые лучше помогут разобраться в факторах - положительно/отрицательно
 # влияющих на учёбу.

In [ ]:
# Гипотеза 1: Пол студента влияет на успеваемость (Т.к. математика является наиболее сложной дисциплиной - проведём исследование на основе результатов по данной дисциплине).
# Гипотеза H0 - Средние баллы по математике у студентов мужского и женского пола статистически не различаются
#               (пол не влияет на успеваемость по математике).

# Гипотеза H1 - Существуют статистически значимые различия в средних баллах по математике между студентами
#               мужского и женского пола (пол влияет на успеваемость по математике).


In [ ]:
# Создадим 2 группы, отличающихся по полу и для удобства оставим только необходимые для анализа столбцы
df_female = df[df['gender'] == 'female']
df_female = df_female[['gender', 'math score']].reset_index()
df_male = df[df['gender'] == 'male']
df_male = df_male[['gender', 'math score']].reset_index()


In [ ]:
print(f'Человек в женской группе: {len(df_female)}')
print(f'Человек в мужской группе: {len(df_male)}')

In [ ]:
# Проверим нормальность распределения с помощью теста Шапиро-Уилка:
# (в зависимости от данной проверки мы сможем определиться с статитстическим
# тестом, который будем использовать для проверги гипотезы)

In [ ]:
stat_male, p_value_male = stats.shapiro(df_male['math score'])
stat_female, p_value_female = stats.shapiro(df_female['math score'])

# Примем границу p-value = 0.05. Если p-value > 0.05,
# то распределение считаем нормальным (не отвергаем нулевую гипотезу)

if p_value_female > 0.05:
  print(f'Распределение нормальное (p_value_female = {p_value_female})')
else:
  print(f'Распределение НЕ нормальное (p_value_female = {p_value_female})')

if p_value_male > 0.05:
  print(f'Распределение нормальное (p_value_male = {p_value_male})')
else:
  print(f'Распределение НЕ нормальное (p_value_male = {p_value_male})')


In [ ]:
# Стратегия проверки гипотезы:
# 1. Тесты на нормальность (Шапиро-Уилка) показали отклонение от нормального
#    распределения в обеих группах (p < 0.05).
# 2. Однако при больших выборках (n ≈ 500 в каждой группе) t-тест Уэлча остается
#    валидным благодаря Центральной Предельной Теореме — он устойчив к нарушению
#    нормальности при n > 30-50.
# 3. Для валидации результатов дополнительно применяем непараметрический тест
#    Манна-Уитни. Согласованность выводов обоих тестов подтвердит надежность
#    статистического заключения.

In [ ]:
# T-тест Уэлча
t_stat, t_pvalue = stats.ttest_ind(df_female['math score'], df_male['math score'], equal_var=False)
print(f"Результат T-тест Уэлча: stat = {t_stat:.2f}, p-value = {t_pvalue:.4f}")

# Тест Манна-Уитни
u_stat, u_pvalue = stats.mannwhitneyu(df_female['math score'], df_male['math score'], alternative='two-sided')
print(f"Результат тест Манна-Уитни: stat = {u_stat:.2f}, p-value = {u_pvalue:.4f}")



In [ ]:
# Сравнение результатов
print(f"Сравниваем результаты тестов:")
if (t_pvalue < 0.05 and u_pvalue < 0.05) or (t_pvalue >= 0.05 and u_pvalue >= 0.05):
    print("Результаты СОВПАДАЮТ - вывод надежный!")
else:
    print("Результаты РАЗЛИЧАЮТСЯ - следует использовать тест Манна-Уитни как более надежный")

In [ ]:
# Вывод: p-value < 0.05, поэтому отвергаем нулевую гипотезу (H0)
# в пользу альтернативной (H1).
# Существуют статистически значимые различия в баллах между мужчинами и женщинами.

In [ ]:
# Несмотря на нарушение нормальности в обеих группах, а также учитывая большой размер выборок (n_male ≈ 500, n_female ≈ 500),
# были применен t-тест Уэлча, который устойчив к отклонениям от нормальности при больших выборках. Для проверки надежности
# результатов дополнительно был проведен тест Манна-Уитни, подтвердивший полученные выводы.

In [ ]:
# Представим наш результат в виде графика, на котором отобразим средний балл по предмету 'math',
# с учётом погрешности, в зависимости от гендерной группы учеников:

In [ ]:
# Создадим функцию для расчёта среднего балла группы учеников, а также
# погрешности, чтобы результат был более точным
def calculate_ci(data, confidence=0.95):
    # confidence = 95% - уровень доверия
    n = len(data)
    mean = np.mean(data)
    sem = stats.sem(data)  # Стандартная ошибка среднего
    margin = sem * stats.t.ppf((1 + confidence) / 2., n-1)  # Вычисляем погрешность
    return mean, margin

In [ ]:
# Рассчитываем среднее с погрешностью для обеих групп
female_mean, female_ci = calculate_ci(df_female['math score'])
male_mean, male_ci = calculate_ci(df_male['math score'])

In [ ]:
print(f'Средний балл девушек: {female_mean}')
print(f'Средний балл юношей: {male_mean}')
print(f'Различие составляет: {abs(female_mean - male_mean)}')

In [ ]:
# Отобразим данные на графике:

In [ ]:
bars = plt.bar(['female', 'male'],
        [female_mean, male_mean],
        color=['#FF9999', '#66B2FF'])

plt.title('Средние баллы по математике')
plt.ylabel('Баллы')

female_and_male_ci = [female_ci, male_ci]
# Проходимся по каждому столбцу и добавляем текст
for i, bar in enumerate(bars):
    # Получаем высоту столбца (значение)
    height = bar.get_height()
    # Добавляем текст над столбцом
    plt.text(
        x=bar.get_x() + bar.get_width() / 2,               # Координата X (середина столбца)
        y=height + 0.5,                                    # Координата Y (чуть выше столбца)
        s=f'{height:.2f} ± {female_and_male_ci[i]:.2f}',   # Текст (среднее значение ± погрешность в данной группе)
        ha='center',                                       # Выравнивание по горизонтали (по центру)
        va='bottom'                                        # Выравнивание по вертикали (снизу)
    )
plt.show()

In [ ]:
# Вывод: Статистически значимые различия обнаружены (p < 0.05).
# Средний балл девушек: 63.63 (c доверительным интервалом = 1.34),
# средний балл юношей: 68.73 (c доверительным интервалом = 1.34)
# Различие составляет 5 баллов, что имеет практическую значимость.

In [ ]:
# Гипотеза 2: Уровень Образование родителя имеет связь с итоговыми балллами на экзаменах.
# Гипотеза H0 - Образование родителя НЕ имеет влиянеи на уровень сдачи экзаменов, учеником.
#               (уровень образования родителя не влияет на успеваемость по дисциплинам).

# Гипотеза H1 - Образование родителя имеет влияние на уровень сдачи эказаменов, учеником.
#               (уровень образования родителя влияет на успеваемость по дисциплинам).

In [ ]:
# Ранее мы уже узнавали, какие есть значения в столбце parental level of education, мы поместили все возможные значения в массив
# education_order, используем этот массив повторно, чтобы сгруппировать данные:

'''
education_order = [
    'high school',
    'some high school',
    'some college',
    "associate's degree",
    "bachelor's degree",
    "master's degree"
]
'''

# Cоздадим словарь со средними баллами для каждой группы
groups_scores = {}
# Заполним его, где ключём будет уровень образования родителя, а значением - Series, содержащий среднее арифметическое студентов по трём дисциплинам
for level in education_order:
    # Фильтруем данные и создаем независимую копию
    group = df[df['parental level of education'] == level].copy()
    # Находим среднее арифметическое
    group['mean score'] = group[['math score', 'reading score', 'writing score']].mean(axis=1)
    # Сохраняем только серию с баллами для последующего статистического теста
    groups_scores[level] = group['mean score']

In [ ]:
# Узнаем размер выборки для каждой группы:
for name_of_group, number_of_element in groups_scores.items():
  print(f'{name_of_group} - размер выборки = {len(number_of_element)}')

In [ ]:
# Проверим нормальность распределения с помощью теста Шапиро-Уилка:
# (в зависимости от данной проверки мы сможем определиться со статистическим
# тестом, который будем использовать для проверки гипотезы)

In [ ]:
# Cоздадим словарь с результатами теста
normality_results = {}
# Создадим цикл, который пробежится по нашему словарю и проверит нормальность распределения данных, после чего - поместит
# результаты теста в словарь
for level, p_value in groups_scores.items():
    stat, p_value_result = stats.shapiro(p_value)
    normality_results[level] = p_value_result

In [ ]:
# Визуализируем результаты теста, в случае, если хоть одно значение p-value <= 0.05, данные распределены НЕ нормально
for level, p_value in normality_results.items():
  if p_value <= 0.05:
    print(f'Группа "{level}" - имеет НЕ нормальное распределение ({p_value} <= 0.05)')
  else:
    print(f'Группа "{level}" - имеет нормальное распределение ({p_value} > 0.05)')
print('----------------------------------------------------')
if any(p <= 0.05 for p in normality_results.values()):
  print('В одной или нескольких группах нарушено нормальное распределение.')
else:
  print('Все группы имеют нормальное распределение.')



In [ ]:
# Итог: для проверки гипотезы нам необходимо работать с большим колличеством
# групп (6) (каждая группа включает выборку размером от 59 до 226 записей) а также не все группы
# имеют нормальное распределение - следовательно, нам потребуется статистический тест, который
# удовлетворяет данным требованиям - используем тест Крускала-Уоллиса (корректная замена ANOVA в тех случаях,
# когда нарушена нормальность распределения данных).

In [ ]:
'''
education_order = [
    'high school',
    'some high school',
    'some college',
    "associate's degree",
    "bachelor's degree",
    "master's degree"
]
'''
groups_data = []  # пустой список, куда будем складывать оценки каждой группы

for edu in education_order:  # перебираем: 'high school', 'some high school', ...
  # 1. Оставляем только строки, где образование == текущее edu
  filtered_df = df[df['parental level of education'] == edu]
  # 2. Берём только столбец с оценками
  scores = filtered_df['math score']
  # 3. Превращаем в массив чисел (вместо Pandas-объекта)
  array_of_scores = scores.values
  # 4. Добавляем этот массив в список
  groups_data.append(array_of_scores)


h_stat, p_value = stats.kruskal(*groups_data)
print(f"Результат теста Краскела-Уоллиса: H = {h_stat:.2f}, p-value = {p_value:.4f}")

if p_value < 0.05:
    print("Различия между группами статистически значимы.")
else:
    print("Различий между группами не обнаружено.")

In [ ]:
# Проведём post-hoc тест, чтобы знать точно - между какими группами обнаружены значения

In [ ]:
# Проводим Dunn's test
posthoc_df = sp.posthoc_dunn(df, val_col='math score', group_col='parental level of education',
                              p_adjust='bonferroni')

print(" РЕЗУЛЬТАТЫ POST-HOC АНАЛИЗА (Dunn's test с поправкой Бонферрони):\n")
display(posthoc_df)

print('----------------------------------------------------')
print("Результаты поиска значимых различий между группами:")

significant_pairs = 0 # Переменная-счётчик значимых резудьтатов
for i in range(len(posthoc_df)):
    for j in range(i+1, len(posthoc_df)):
        p_val = posthoc_df.iloc[i, j]
        if p_val < 0.05:
            group1 = posthoc_df.columns[i]
            group2 = posthoc_df.columns[j]
            significant_pairs += 1
            print(f"{group1} и {group2}: p = {p_val:.4f}")

print('----------------------------------------------------')
if significant_pairs >= 1:
  print(f"Найдено {significant_pairs} значимых различий между группами.")
else:
  print(f"Найдено {significant_pairs} значимых различий между группами.")

In [ ]:
# Вывод: p-value < 0.05, поэтому отвергаем нулевую гипотезу (H0)
# в пользу альтернативной (H1).
# Результаты показывают, что значимые различия есть между группами high school и группами с более высоким
# образованием (associate's degree, bachelor's degree, master's degree, some college) — что логично: дети
# родителей с низким уровнем образования имеют статистически значимо более низкий средний балл.

In [ ]:
# Выизуализируем результаты для наглядности

# Рассчитываем средние и доверительные интервалы
means, cis = [], []
for value in groups_scores.values():
  a, b = calculate_ci(value)
  means.append(a)
  cis.append(b)

# Выводим график
fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.bar(education_order, means, yerr=cis, capsize=5,
              color='skyblue', edgecolor='black', alpha=0.8)
labels = [f'{m:.1f} ± {ci:.1f}' for m, ci in zip(means, cis)]
ax.bar_label(bars, labels = labels, padding=0, fontweight='bold')


ax.set_ylabel('Средний балл')
ax.set_title('Влияние образования родителей на успеваемость (с доверительными интервалами)')
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Вывод: Статистически значимые различия обнаружены (p < 0.0001).
# Средний балл ученика, чей родитель имеет уровень образования:
# high school - составляет 63.1 (c доверительным интервалом = 1.9),
# Средний балл ученика, чей родитель имеет уровень образования:
# master's degree - составляет 73.6 (c доверительным интервалом = 3.5),
# Различие составляет 10.5 баллов, что имеет практическую значимость.
# С увеличением уровня ступени образования родителя - растёт и
# средний балл ребёнка.

In [ ]:
# Гипотеза 3:
'''
H0 (Нулевая): Подготовительный курс дает одинаковую (или меньшую)
относительную пользу для студентов с льготным питанием (free/reduced)
по сравнению со студентами со стандартным питанием (standard).
H1 (Альтернативная): Подготовительный курс дает статистически значимо
больший относительный прирост баллов для студентов с льготным питанием
(free/reduced), чем для студентов со стандартным питанием (standard).
'''

In [ ]:
# Создадим отдельную колонку в нашей основной таблице, в которой будет
# средний балл ученика (среднее арифметическое результатов по всем тестам)
df['average_score'] = df[['math score', 'reading score', 'writing score']].mean(axis=1)

In [ ]:
'''
В данной гипотезе мы исследуем влияние двух факторов (тип питания и
подготовительный курс) на средний балл.
У нас есть 4 независимых группы:
1) free/reduced + completed
2) free/reduced + none
3) standard + completed
4) standard + none

Если все 4 группы нормальны и дисперсии равны - используем Two-way ANOVA
для оценки общего эффекта факторов

Если хоть одно условие нарушено - используем тест Крускала-Уоллиса +
тест Манна-Уитни для попарных сравнений (непараметрические методы)
'''

In [ ]:
# Группируем данные
groups = df.groupby(['lunch', 'test preparation course'])['average_score']

# Проверим нормальность распределения с помощью теста Шапиро-Уилка:
for name, group in groups:
    stat, p = stats.shapiro(group)
    print(f'Группа {name}: W={stat:.3f}, p={p:.3f} -> {"Нормальное" if p > 0.05 else "Не нормальное"}')

# Проверяем равенство дисперсий с помощью тест Левена
group_list = [group for name, group in groups]
stat, p_levene = stats.levene(*group_list)
print(f'\nТест Левена: p={p_levene:.3f} -> {"Дисперсии равны" if p_levene > 0.05 else "Дисперсии разные"}')

In [ ]:
# Вывод: Не все группы распределены нормально, несмотря на то, что дисперсии равны
# - одно из условий не соблюдено (группа ('free/reduced', 'none'): p=0.031; 0.031 < 0.05),
# используем непараметрические методы для дальнейшего анализа.

In [ ]:
# 1) Проводим тест Крускала-Уоллиса

In [ ]:
# Извлекаем данные по группам
free_completed = df[(df['lunch']=='free/reduced') & (df['test preparation course']=='completed')]['average_score']  # Одинарные квадратные скобки ['average_score'] после фильтрации возвращают Series
free_none = df[(df['lunch']=='free/reduced') & (df['test preparation course']=='none')]['average_score']
std_completed = df[(df['lunch']=='standard') & (df['test preparation course']=='completed')]['average_score']
std_none = df[(df['lunch']=='standard') & (df['test preparation course']=='none')]['average_score']

# Проводим тест Крускала-Уоллиса
h_stat, p_kruskal = stats.kruskal(free_completed, free_none, std_completed, std_none)
print(f"H-статистика: {h_stat:.3f}")
print(f"p-value: {p_kruskal:.3f}")
if p_kruskal < 0.05:
  print('Различия между группами статистически значимы')
else:
    print('Различия между группами не значимы')

In [ ]:
# 2) Проводим попарное сравнение (Тест Манна-Уитни)

In [ ]:
print("Эффект подготовительного курса для группы free/reduced:")
u_stat1, p_free = stats.mannwhitneyu(free_completed, free_none, alternative='greater')
print(f"U-статистика: {u_stat1:.3f}")
print(f"p-value: {p_free:.3f}")
if p_free < 0.05:
  print('Курс значимо улучшает результаты (p-value < 0.05)')
else:
    print('Курс не даёт значимого улучшения (p-value >= 0.05)')
print('----------------------------------------------------')
print("Эффект подготовительного курса для группы standart:")
u_stat2, p_std = stats.mannwhitneyu(std_completed, std_none, alternative='greater')
print(f"U-статистика: {u_stat2:.3f}")
print(f"p-value: {p_std:.3f}")
if p_std < 0.05:
  print('Курс значимо улучшает результаты (p-value < 0.05)')
else:
    print('Курс не даёт значимого улучшения (p-value >= 0.05)')

In [ ]:
# 3) Посмотрим на величину эффекта

In [ ]:
median_free_completed = free_completed.median()
median_free_none = free_none.median()
median_std_completed = std_completed.median()
median_std_none = std_none.median()

effect_free = median_free_completed - median_free_none
effect_std = median_std_completed - median_std_none
difference = effect_free - effect_std

print(f"free/reduced: {median_free_completed:.2f} - {median_free_none:.2f} = {effect_free:.2f} баллов")
print('----------------------------------------------------')
print(f"standard: {median_std_completed:.2f} - {median_std_none:.2f} = {effect_std:.2f} баллов")
print('----------------------------------------------------')
print(f"Разница в эффекте: {difference:.2f} баллов")
if difference > 0:
  print('Курс подготовки более эффективен для группы free/reduced')
else:
  print('Курс подготовки более эффективен для группы standard')


In [ ]:
df_not_clean

In [ ]:
# Вывод: Подготовительный курс статистически значимо улучшает результаты тестов,
# при этом эффект сильнее для социально незащищённых студентов (free/reduced).

#Task 4 - Рекомендации

На основе проведенного статистического анализа были проверены ключевые гипотезы, влияющие на успеваемость студентов. Ниже представлены подтвержденные выводы и конкретные шаги для учебного заведения / образовательной платформы.
1. Резюме проверенных гипотез
Гипотеза 1: Студенты, прошедшие подготовительный курс (test preparation course), показывают более высокие результаты экзаменов, чем те, кто его не проходил.
✅ Статус: Подтверждена. Статистические тесты показали статистически значимое превосходство группы, прошедшей подготовку, по всем трем предметам (математика, чтение, письмо).
Гипотеза 2: Уровень образования родителей влияет на академические результаты студентов.
✅ Статус: Подтверждена. Непараметрический дисперсионный анализ (Крускала-Уоллиса) и пост-хок тест Данна с поправкой Бонферрони выявили статистически значимое отставание студентов, чьи родители имеют только школьное образование (high school), по сравнению со студентами из семей с более высоким уровнем образования.
Гипотеза 3: Подготовительный курс дает статистически значимо больший относительный прирост баллов для студентов с льготным питанием (free/reduced), чем для студентов со стандартным питанием (standard).
✅ Статус: Подтверждена. Анализ показал, что подготовительные курсы особенно эффективны для студентов из менее обеспеченных семей. Это означает наличие взаимодействия факторов (interaction effect): социальный статус усиливает эффект от образовательных интервенций.

💡 2. Практические рекомендации для бизнеса / администрации
Приоритетное предоставление курсов социально уязвимым группам (Evidence-Based Equity)
На основе подтверждения Гипотезы 3, рекомендуется внедрить систему приоритетного доступа к подготовительным курсам для студентов, имеющих право на льготное питание. Эти студенты демонстрируют наибольший относительный прогресс, что означает максимальную отдачу от инвестиций (ROI) именно в эту группу. Это не только вопрос социальной справедливости, но и эффективное распределение ресурсов.
Целевая программа поддержки уязвимых групп (Targeted Intervention)
Выявленный разрыв в результатах между студентами из семей с разным уровнем образования родителей (Гипотеза 2) требует компенсирующих мер. Рекомендуется внедрить программу академического менторства именно для студентов, чьи родители имеют уровень образования high school.
Комплексный подход: курсы + питание
Поскольку оба фактора (подготовительный курс и тип питания) статистически значимо влияют на результаты, рекомендуется рассмотреть интегрированную программу поддержки, которая объединяет:
Бесплатные подготовительные курсы
Обеспечение качественным питанием
Академическое консультирование
Такая программа может иметь синергетический эффект и максимально сократить образовательное неравенство.
Информационная кампания для студентов и родителей
Запустить рассылку или презентацию, которая наглядно демонстрирует эффективность подготовительных курсов, особенно для студентов из семей с разным социально-экономическим статусом.